# Script 1

#### Objective
The next notebook takes info from CVs, transform this info to it's vector embedding representation, feeds this data into a database and allows to search in the database the CVs that suits the most to a query.

### 1. Initializing a local client

Qdrant db client can work in the cloud or locally in three ways: In-memory, in hard drive and in a docker container (Prefered for production-ready projects). Here "In-memory" mode will be used for testing purposes.

To make a quick test for the embedding model, a simple embedding model is used the the in-memory feature is used for Qdrant. ":memory:" Allows Qdrant to work in in-memory mode, so the database lives in my PC's RAM.  One can work with the database as long as the notebook kernel is not restarted, or the code snippet down below is not re-ran

In [2]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

### 2. Creating collections

A qdrant collection is the fundamental piece of information of the vectorial database. It's an isolated container containing the vectors, IDs and metadata (data points). It's the equivalent of a Table in a SQL database.

In [3]:
models_to_test = {
    "harrier-oss-v1-0.6b": {"size": 1024, "model_name": "microsoft/harrier-oss-v1-0.6b"},
}

# Create a unique collection for each model
for name, config in models_to_test.items():
    COLLECTION_NAME = f"test_{name}"
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=config["size"],
            distance=models.Distance.COSINE
        )
    )
print("Collections created successfully!")

Collections created successfully!


### 3. Loading CV data and embedding the data

Qdrant cloud has something called "Cloud Inference" in which the data is converted to embeddings with a fixed model. However, for Qdrant local, we have to do that process ourselves (Unless we use fastembed), but we are free to use any model, which is exactly what I want for this and script 2. The thing with fastembed is that for models that are not default, additional configurations must be done, which is not ideal for this case, so I'll just use Huggin Face's sentence-transformers

In [4]:
import json

with open("../data/cv_extracted_info_eng.json", "r") as file:
    cv_data = json.load(file)

print(cv_data)

[{'name': 'CRISTIAN BENJAMIN GARCÍA CASIERRA', 'email': 'cristian_garcia@outlook.com', 'linkedin': '', 'phone': '+57 3160546495', 'about_me': "Electronics engineer with three years of experience in electronic circuit design, three years in FM and AM radio transmitter maintenance, and three years as a Flutter developer. Proficient in programming languages such as Dart, HTML, Python, CSS, JavaScript, and frameworks like React and Angular, with knowledge of UX/UI. I constantly strive to push my limits and learn new ways to apply what I've learned. I can work both individually and as part of a team, and I also possess strong communication skills to convey my ideas effectively.", 'profession': 'Electronics Engineer, Front End Developer, Project Engineer, Maintenance Technician', 'seniority_level': 'Senior', 'location': 'Colombia', 'experience_years': '20+ years', 'certifications': ['ADVANCED FLUTTER: TAKE YOUR KNOWLEDGE TO THE NEXT LEVEL UDEMY (MAR. 2025 - SEP. 2025)'], 'skills': ['CSS', 'J

Loading the vector embedding model

In [5]:
import os
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

# Load environment variables from the .env file in the root directory
load_dotenv("../.env")
HUGGING_FACE_API_KEY = os.getenv("HUGGING_FACE_API_KEY")

# Load reference model for token limit constraints (smallest model: GIST-all-MiniLM-L6-v2 / All-MiniLM-L6-v2)
ref_model_name = "avsolatorio/GIST-all-MiniLM-L6-v2"
ref_model = SentenceTransformer(ref_model_name, token=HUGGING_FACE_API_KEY, trust_remote_code=True)

# Model for encoding
model = SentenceTransformer(models_to_test["harrier-oss-v1-0.6b"]["model_name"], token=HUGGING_FACE_API_KEY, trust_remote_code=True)


/Users/col-ae-068/Documents/personal-projects/ai-eng-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 7609.58it/s]


Creating the chunks

In [6]:
def is_less_than_max_tokens(ref_model, text, applicant_name, chunk_num):
    num_tokens = len(ref_model.tokenizer.encode(text))
    max_tokens = ref_model.max_seq_length
    if num_tokens > max_tokens:
        return False
    return True

max_tokens = ref_model.max_seq_length

data_points = []
for idx, cv in enumerate(cv_data):
    # Fields like "name", "profession", "email", "linkedin" and "phone" can go in the metadata
    
    # First CV chunk
    chunk = (
        f"Candidate name: {cv['name']} | "
        f"Candidate profession: {cv['profession']} | "
        f"Content: (about_me: {cv['about_me']}, seniority_level: {cv['seniority_level']}, location: {cv['location']}, experience_years: {cv['experience_years']}, languages: {cv.get('languages', [])})"
    )
    if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 1)):
        data_points.append({
            "chunk": chunk,
            "chunk_meta": "about_me-seniority_level-location-experience_years-languages", # unique identifier of the chunk
            "cv": cv
        })
    else:
        pass # This will never happen

    # Second CV chunk
    chunk = (
        f"Candidate name: {cv['name']} | "
        f"Candidate profession: {cv['profession']} | "
        f"Content: (certifications: {cv['certifications']}, education: {cv['education']}, skills: {cv['skills']})"
    )
    if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 2)):
        data_points.append({
            "chunk": chunk,
            "chunk_meta": "certifications-education-skills", # unique identifier of the chunk
            "cv": cv
        })
    else:
        # Separate into three chunks
        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (certifications: {cv['certifications']})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 2.1)):
            data_points.append({
            "chunk": chunk,
            "chunk_meta": "certifications", # unique identifier of the chunk
            "cv": cv
        })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 2.1 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Attaching anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "certifications", # unique identifier of the chunk
                "cv": cv
            })

        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (education: {cv['education']})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 2.2)):
            data_points.append({
            "chunk": chunk,
            "chunk_meta": "education", # unique identifier of the chunk
            "cv": cv
        })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 2.2 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Appending data anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "education", # unique identifier of the chunk
                "cv": cv
            })

        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (skills: {cv['skills']})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 2.3)):
            data_points.append({
            "chunk": chunk,
            "chunk_meta": "skills", # unique identifier of the chunk
            "cv": cv
        })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 2.3 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Appending data anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "skills", # unique identifier of the chunk
                "cv": cv
            })

    # Third chunk
    chunk = (
        f"Candidate name: {cv['name']} | "
        f"Candidate profession: {cv['profession']} | "
        f"Content: (experience: {cv['experience']})"
    )
    if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 3)):
        data_points.append({
            "chunk": chunk,
            "chunk_meta": "experience", # unique identifier of the chunk
            "cv": cv
        })
    else:
        # Split experience in half
        exp = f"experience_1: {cv['experience']}"
        half = int(len(exp)/2)
        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (experience_part_1: {exp[:half]})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 3.1)):
            data_points.append({
            "chunk": chunk,
            "chunk_meta": "experience_part_1", # unique identifier of the chunk
            "cv": cv
        })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 3.1 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Appending data anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "experience_part_1", # unique identifier of the chunk
                "cv": cv
            })

        chunk = (
            f"Candidate name: {cv['name']} | "
            f"Candidate profession: {cv['profession']} | "
            f"Content: (experience_part_2: {exp[half:]})"
        )
        if (is_less_than_max_tokens(ref_model, chunk, cv['name'], 3.2)):
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "experience_part_2", # unique identifier of the chunk
                "cv": cv
            })
        else:
            num_tokens = len(ref_model.tokenizer.encode(chunk))
            print(f"serialized CV for {cv['name']}, chunk 3.2 has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}). Appending data anyways...")
            data_points.append({
                "chunk": chunk,
                "chunk_meta": "experience_part_2", # unique identifier of the chunk
                "cv": cv
            })

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (561 > 512). Running this sequence through the model will result in indexing errors


serialized CV for JUAN SANTIAGO VILLEGAS LÓPEZ, chunk 2.1 has 514 tokens, which is greater than the number of tokens this model supports (512). Attaching anyways...
serialized CV for Deivy Stiven Hernandez Castañeda, chunk 3.1 has 519 tokens, which is greater than the number of tokens this model supports (512). Appending data anyways...
serialized CV for Deivy Stiven Hernandez Castañeda, chunk 3.2 has 525 tokens, which is greater than the number of tokens this model supports (512). Appending data anyways...


Vector embedding the CVs

In [7]:
embeddings = model.encode([point["chunk"] for point in data_points])
print(f"embeddings.shape: {embeddings.shape}")
embeddings = embeddings.tolist()

embeddings.shape: (303, 1024)


### 4. Inserting the data to the database

In [8]:
EMBEDDING_MODEL = models_to_test["harrier-oss-v1-0.6b"]['model_name']

client.upload_points(
    collection_name=COLLECTION_NAME,
    points=[
        models.PointStruct(
            id=idx,
            vector=embeddings[idx],
            # Store the original dictionary (to keep queryable fields) and the chunked text (useful for LLM context later)
            payload={
                **data_point['cv'],
                "chunk": data_point['chunk']
            }
        )
        for idx, data_point in enumerate(data_points)
    ],
)

### 5. Querying to the database

Loading some job descriptions

In [9]:
with open("../data/job_descriptions_train_batch.json", "r", encoding="utf-8") as f:
    job_descriptions = json.load(f)

job_descriptions

[{'id': 'jd_ai_agents_engineer',
  'title': 'Founding AI Agents Engineer & Automations Developer',
  'description': 'Senior hybrid role in Medellín ($5M–7M COP/month) with 5+ years experience. Responsible for designing, deploying, and maintaining autonomous AI agents and low-code digital workflows using n8n, LangGraph, CrewAI, MCP, RAG, Python, and LLMs (GPT-4, Claude 3.5, Gemini). Connects APIs with legacy CRMs (Bitrix24, Zoho) and establishes productized SaaS formulas. Requires fluent technical English.',
  'source': 'AI Agents Developer & AI Strategist (Founding Team) _ The 100x Company _ LinkedIn.pdf'},
 {'id': 'jd_btl_marketing_coordinator',
  'title': 'BTL Marketing Coordinator',
  'description': 'Mid-level role in Medellín ($7.2M COP/month, 3 yrs exp) designing and executing national BTL strategies and brand events. Responsibilities include approving distributor proposals, overseeing advertising merchandising, coordinating event logistics, managing BTL auxiliary staff, and manag

Testing the filtering. The first job description says the applicant must live in Medellin. So first search will be done without filtering and the second one will be done with filtering

Loading the ground truth matrix to check the scores given by the LLM in these top K CVs for the specific job description

In [10]:
with open("../data/ground_truth_matrix.json", "r", encoding="utf-8") as f:
    ground_truth_matrix = json.load(f)

Indexing the two fields to be used for filtering

In [11]:
fields_to_index = ["location", "seniority_level"]

for field in fields_to_index:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field,
        field_schema=models.PayloadSchemaType.TEXT
    )

/var/folders/fq/ybvfzjt96mz_ng7wrlmkxpkh0000gn/T/ipykernel_48270/3315671777.py:4: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


Comparing multiple filters

In [12]:
import math
import pandas as pd
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(models_to_test["harrier-oss-v1-0.6b"]["model_name"], token=HUGGING_FACE_API_KEY)
Q = len(job_descriptions)
hr_thr = 2 # Hit rate relevance threshold
mrr2_thr = 2 # MRR_2 relevance threshold
mrr3_thr = 3 # MRR_3 relevance threshold
top_k = 10

metrics = {
    "hit_rate": {},
    "mrr_2": {}, # Mean Reciprocal Rank with a relevance (Benchmark score) of 2
    "mrr_3": {},
    "ndcg_10": {}, # Normalized Discounted Cumulative Gain at K=10
}

filters_to_test = {
    "no_filter": None,
    "by_location": models.Filter(
        must=[
            # MatchText handles "Medellin", "Medellín", "medellin", etc. automatically
            models.FieldCondition(
                key="location",
                match=models.MatchText(text="medellin")
            )
        ]
    ),
    "by_seniority": models.Filter(
        must=[
            models.FieldCondition(
                key="seniority_level",
                match=models.MatchText(text="Senior")
            )
        ]
    ),
    "by_location_n_seniority": models.Filter(
        must=[
            # MatchText handles "Medellin", "Medellín", "medellin", etc. automatically
            models.FieldCondition(
                key="location",
                match=models.MatchText(text="medellin")
            ),
            models.FieldCondition(
                key="seniority_level",
                match=models.MatchText(text="Senior")
            )
        ]
    )
}

for filter_name, filter_config in filters_to_test.items():

    hit_rate_accum = 0
    mrr2_accum = 0
    mrr3_accum = 0
    ndcg_accum = 0.0

    print("\n" + "="*80)
    print(f" Filter Name: {filter_name}")
    print("="*80)

    for job in job_descriptions:
        
        query_vector = model.encode(job["description"]).tolist()
        
        result = client.query_points_groups(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            query_filter=filter_config,
            limit=top_k,
            group_by="email",
            group_size=1
        )

        has_hit = False
        mrr2_rr = 0.0
        mrr3_rr = 0.0
        dcg = 0.0

        for rank, group in enumerate(result.groups, start=1):
            eval_matches = [e for e in ground_truth_matrix[job["id"]]["evaluations"] if e["email"] == group.id]
            if eval_matches:
                benchmark_score = eval_matches[0]["score"]
                print(f"score {benchmark_score} in job {job["id"]}")
                
                # Calculate DCG for current result
                dcg += benchmark_score / math.log2(rank + 1)
                
                if benchmark_score >= hr_thr:
                    has_hit = True
                
                if benchmark_score >= mrr2_thr and mrr2_rr == 0.0:
                    mrr2_rr = 1.0 / rank
                    
                if benchmark_score >= mrr3_thr and mrr3_rr == 0.0:
                    mrr3_rr = 1.0 / rank

        # Ideal DCG (IDCG) for current job from ground truth
        all_eval_scores = sorted(
            [e["score"] for e in ground_truth_matrix[job["id"]]["evaluations"]],
            reverse=True
        )[:top_k]
        idcg = sum(score / math.log2(rank + 1) for rank, score in enumerate(all_eval_scores, start=1))
        
        ndcg_query = (dcg / idcg) if idcg > 0 else 0.0
        ndcg_accum += ndcg_query

        if has_hit:
            hit_rate_accum += 1
            
        mrr2_accum += mrr2_rr
        mrr3_accum += mrr3_rr

    metrics["hit_rate"][filter_name] = hit_rate_accum / Q
    metrics["mrr_2"][filter_name] = mrr2_accum / Q
    metrics["mrr_3"][filter_name] = mrr3_accum / Q
    metrics["ndcg_10"][filter_name] = ndcg_accum / Q

df_metrics = pd.DataFrame(metrics)
df_metrics


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 7790.50it/s]



 Filter Name: no_filter
score 2 in job jd_ai_agents_engineer
score 1 in job jd_ai_agents_engineer
score 3 in job jd_ai_agents_engineer
score 1 in job jd_ai_agents_engineer
score 3 in job jd_ai_agents_engineer
score 3 in job jd_ai_agents_engineer
score 2 in job jd_ai_agents_engineer
score 1 in job jd_ai_agents_engineer
score 1 in job jd_ai_agents_engineer
score 2 in job jd_ai_agents_engineer
score 3 in job jd_btl_marketing_coordinator
score 2 in job jd_btl_marketing_coordinator
score 3 in job jd_btl_marketing_coordinator
score 1 in job jd_btl_marketing_coordinator
score 2 in job jd_btl_marketing_coordinator
score 1 in job jd_btl_marketing_coordinator
score 2 in job jd_btl_marketing_coordinator
score 1 in job jd_btl_marketing_coordinator
score 0 in job jd_btl_marketing_coordinator
score 0 in job jd_btl_marketing_coordinator
score 1 in job jd_senior_sap_data_analyst
score 1 in job jd_senior_sap_data_analyst
score 1 in job jd_senior_sap_data_analyst
score 1 in job jd_senior_sap_data_analy

,hit_rate,mrr_2,mrr_3,ndcg_10
no_filter,1.0,0.833333,0.50,0.756539
by_location,0.0,0.000000,0.00,0.000000
by_seniority,1.0,0.850000,0.45,0.713979
by_location_n_seniority,0.0,0.000000,0.00,0.000000
